# Séance 4 — Ingénierie des variables (couche Gold)
*Session 4 — Feature engineering (Gold layer)*

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noe2019/fraude-360-cours/blob/main/cours/laboratoires/04_variables_gold.ipynb)

**Cours :** Science des données appliquée : détection de la fraude transactionnelle (45 h) · **Chargé de cours :** Noé Carème Fouotsa Manfouo · **Durée :** 3 h

| Parcours de formation | Cette séance |
|---|---|
| Gestion — UdeS École de gestion (SIMQG), Bishop's Williams School | ○ démonstration |
| Informatique — UdeS maîtrise/bac en informatique (IA et science des données) | ● séance complète |
| Cégep / formation continue — Cégep de Sherbrooke (AEC), UdeS formation continue | ○ démonstration |

**À la fin de cette séance, vous saurez :**

1. construire des variables de vélocité avec des fenêtres temporelles glissantes ;
2. calculer des distances géographiques (formule de haversine) et des vitesses de déplacement ;
3. vérifier qu'une variable n'utilise **que** l'information disponible au moment de la décision ;
4. relier une variable à son pouvoir discriminant.

**Mode d'emploi :** exécutez les cellules dans l'ordre (`Maj + Entrée`). Complétez les cellules `# VOTRE CODE ICI`, puis
lancez la cellule ✅ qui suit pour obtenir une rétroaction immédiate. Chaque exercice indique le ou les parcours concernés :
🟢 tous · 🟣 Gestion + Informatique · 🔷 Informatique · 🔶 Gestion. Les exercices 🔷/🔶 sont facultatifs pour les autres parcours.

In [ ]:
# --- Configuration (une fois par session) ----------------------------------------
import sys, pathlib, subprocess, shutil
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    if not pathlib.Path("fraude-360-cours").exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/noe2019/fraude-360-cours.git"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "fraude-360-cours/cours/requirements-colab.txt"], check=True)
    if shutil.which("java") is None:   # PySpark a besoin d'un Java 11, 17 ou 21
        subprocess.run("apt-get -qq install -y openjdk-17-jre-headless > /dev/null", shell=True, check=True)
    RACINE = pathlib.Path("fraude-360-cours").resolve()
else:
    ici = pathlib.Path.cwd().resolve()
    RACINE = ici.parents[1] if ici.name == "laboratoires" else ici
sys.path.insert(0, str(RACINE / "cours"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from outils_cours import preparer, config, table, verifier, verifier_proche, image, lire_json
pd.set_option("display.max_columns", 40)
print("Racine du projet :", RACINE)

Quatre familles de variables, calculées **pour chaque transaction, avec le passé seulement** :

| Famille | Exemples | Typologie visée |
|---|---|---|
| Vélocité de la carte | `nb_tx_carte_1h`, `montant_somme_carte_24h`, `nb_micro_tx_carte_1h` | test de carte, carte volée |
| Géographie | `distance_domicile_km`, `vitesse_kmh` (déplacement impossible) | carte clonée |
| Écart au comportement | `ratio_montant_moyenne_carte`, `est_nouveau_marchand_carte` | usurpation |
| Réputation du marchand | `taux_fraude_marchand_30j` | marchands compromis |

En Spark, une fenêtre `Window.partitionBy("carte_id").orderBy("ts_unix").rangeBetween(-3600, 0)` compte les transactions
de la même carte dans l'heure qui précède.

In [ ]:
cfg = preparer("variables")
spec = lire_json(cfg.chemin("artefacts") / "variables" / "specification_courante.json")
print(len(spec["variables_numeriques"]), "variables numériques,", len(spec["variables_categorielles"]), "catégorielles")
gold = table("gold", "caracteristiques")
gold[["carte_id", "horodatage", "montant", "nb_tx_carte_1h", "nb_micro_tx_carte_1h", "vitesse_kmh", "est_fraude"]].head()

In [ ]:
import inspect
from fraude360.features import gold as module_gold
print("\n".join(inspect.getsource(module_gold.variables_de_base).splitlines()[:30]))

### Exercice 4.1 — 🟢 Tous les parcours

Écrivez `haversine(lat1, lon1, lat2, lon2)` (en km, rayon terrestre 6 371 km), compatible avec des tableaux NumPy.
Testez-la entre Sherbrooke (45,4042; −71,8929) et Montréal (45,5019; −73,5674).

In [ ]:
# VOTRE CODE ICI
def haversine(lat1, lon1, lat2, lon2):
    ...
raise NotImplementedError()

In [ ]:
# ✅ Vérification (ne pas modifier)
verifier(128 < haversine(45.4042, -71.8929, 45.5019, -73.5674) < 134, "≈ 131 km à vol d'oiseau")
verifier(np.allclose(haversine(np.array([0.0]), np.array([0.0]), np.array([0.0]), np.array([0.0])), 0))

### Exercice 4.2 — 🔷 Informatique — approfondissement

Recalculez `nb_tx_carte_1h` **en pandas** : pour chaque transaction, le nombre de transactions de la même carte
dans les 3 600 secondes précédentes, transaction courante incluse. Stockez le résultat dans `gold["nb_tx_1h_pandas"]`.
Indice : trier par `horodatage`, `groupby("carte_id")`, puis `.rolling("3600s", on="horodatage", closed="both")`.

In [ ]:
# VOTRE CODE ICI
gold["nb_tx_1h_pandas"] = ...
raise NotImplementedError()

In [ ]:
# ✅ Vérification (ne pas modifier)
_accord = (gold["nb_tx_1h_pandas"] == gold["nb_tx_carte_1h"]).mean()
print(f"Accord avec Spark : {_accord:.2%}")
verifier(_accord > 0.99, "Au moins 99 % d'accord (les égalités de secondes peuvent différer)")

### Exercice 4.3 — 🟢 Tous les parcours

Calculez `taux_par_micro` : le taux de fraude selon `nb_micro_tx_carte_1h` regroupé en classes `0`, `1`, `2`, `3+`
(utilisez `pd.cut` ou `clip(upper=3)`). Que remarquez-vous ?

In [ ]:
# VOTRE CODE ICI
taux_par_micro = ...
raise NotImplementedError()

In [ ]:
# ✅ Vérification (ne pas modifier)
verifier(taux_par_micro.loc["3+"] > 10 * taux_par_micro.loc["0"], "Les rafales de micro-montants signalent le test de carte")

**❓ Question (🔷 Informatique — approfondissement) —** `taux_fraude_marchand_30j` utilise les étiquettes des 30 derniers jours, jusqu'à la seconde précédente. Or une fraude n'est confirmée qu'après ~7 jours. Y a-t-il une **fuite d'information** ? Comment corriger la fenêtre ?

*Votre réponse ici.*

## Bilan

- Les variables de comportement se calculent avec des fenêtres glissantes **sur le passé uniquement**.
- Chaque variable vise une typologie : c'est ce qui les rend explicables aux enquêteurs.
- Toujours vérifier la disponibilité de l'information au moment de la décision (fuites d'étiquettes).

**Pour aller plus loin :**

- Feature Store Databricks / Feast : servir les mêmes variables en ligne et hors ligne.

---
*Fraude-360 — données entièrement synthétiques. Cours conçu par Noé Carème Fouotsa Manfouo · licence MIT (code) et CC BY 4.0 (contenu pédagogique).*